## **NSE Bulk daily updates**

In [4]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_bulk_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)
os.makedirs("/lakehouse/default/Files/data/raw/logs", exist_ok=True)

# today date
today = datetime.today().strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"bulkdeals_{today}.csv"
)

try:

    # session + headers
    session = requests.Session()

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip, deflate"
    }

    # get cookies
    session.get(
        "https://www.nseindia.com",
        headers=headers
    )

    # API URL
    url = (
        "https://www.nseindia.com/api/"
        "historicalOR/bulk-block-short-deals"
        f"?optionType=bulk_deals"
        f"&from={today}"
        f"&to={today}"
        f"&csv=true"
    )

    response = session.get(
        url,
        headers=headers
    )

    # read csv
    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # clean columns
    df.columns = (
        df.columns
        .str.replace("ï»¿", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.strip()
    )

    # save file
    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    status = "SUCCESS"
    rows = len(df)

    if rows == 0:
        message = "No data yet"
    else:
        message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# simple log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_bulk",
    "status": status,
    "rows": rows,
    "message": message
}])


# append log
if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

SUCCESS
Rows: 0
No data yet


## **NSE Block deals daily update**

In [9]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_block_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)
os.makedirs("/lakehouse/default/Files/data/raw/logs", exist_ok=True)

# today date
# today = datetime.today().strftime("%d-%m-%Y")
today = "29-05-2026"
file_path = os.path.join(
    save_dir,
    f"blockdeals_{today}.csv"
)

try:

    # session + headers
    session = requests.Session()

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip, deflate"
    }

    # get cookies
    session.get(
        "https://www.nseindia.com",
        headers=headers
    )

    # API URL
    url = (
        "https://www.nseindia.com/api/"
        "historicalOR/bulk-block-short-deals"
        f"?optionType=block_deals"
        f"&from={today}"
        f"&to={today}"
        f"&csv=true"
    )

    response = session.get(
        url,
        headers=headers
    )

    # read csv
    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # clean columns
    df.columns = (
        df.columns
        .str.replace("ï»¿", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.strip()
    )

    # save file
    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    status = "SUCCESS"
    rows = len(df)

    if rows == 0:
        message = "No data yet"
    else:
        message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_block",
    "status": status,
    "rows": rows,
    "message": message
}])


# append log
if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

SUCCESS
Rows: 3433
File saved


## **BSE Bulk Deals Daily Update**

In [7]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_bulk_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today
today = datetime.today()

api_date = today.strftime("%d/%m/%Y")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"bulk_{file_date}.csv"
)

try:

    url = (
        "https://api.bseindia.com/BseIndiaAPI/api/"
        "BulknBlockBETADwnld/w"
        f"?DealType=1"
        "&sc_code="
        f"&FDate={api_date}"
        f"&TDate={api_date}"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # standardize column names
    df.columns = [
        "Deal_Date",
        "Security_Code",
        "Company",
        "Client_Name",
        "Deal_Type",
        "Quantity",
        "Price"
    ]

    # save
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# logging
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_bulk",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

SUCCESS
Rows: 65
File saved


## **BSE Block Daily Update**

In [9]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_block_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today
today = datetime.today()

api_date = today.strftime("%d/%m/%Y")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"block_{file_date}.csv"
)

try:

    url = (
        "https://api.bseindia.com/BseIndiaAPI/api/"
        "BulknBlockBETADwnld/w"
        f"?DealType=2"
        "&sc_code="
        f"&FDate={api_date}"
        f"&TDate={api_date}"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    # no block deals today
    if response.text.strip() == "":

        df = pd.DataFrame()

    else:

        df = pd.read_csv(
            io.StringIO(response.text)
        )

        # standardize column names
        df.columns = [
            "Deal_Date",
            "Security_Code",
            "Company",
            "Client_Name",
            "Deal_Type",
            "Quantity",
            "Price"
        ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)

    if rows == 0:
        message = "No data yet"
    else:
        message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# logging
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_block",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

SUCCESS
Rows: 0
No data yet


## **NSE Trade daily update**

In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_trade_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today's date
today = datetime.today()

date_str = today.strftime("%Y%m%d")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"trade_{file_date}.csv"
)

try:

    url = (
        "https://nsearchives.nseindia.com/content/cm/"
        f"BhavCopy_NSE_CM_0_0_0_{date_str}_F_0000.csv.zip"
    )

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=20
    )

    if response.status_code != 200:
        raise Exception("No data yet")

    # read bhavcopy
    zip_file = zipfile.ZipFile(
        io.BytesIO(response.content)
    )

    csv_name = zip_file.namelist()[0]

    df = pd.read_csv(
        zip_file.open(csv_name)
    )

    # keep NSE cash market stocks only
    df = df[
        (df["Sgmt"] == "CM") &
        (df["FinInstrmTp"] == "STK")
    ]

    # keep required columns
    df = df[[
        "TradDt",
        "TckrSymb",
        "OpnPric",
        "HghPric",
        "LwPric",
        "ClsPric",
        "TtlTradgVol"
    ]]

    # rename columns
    df.columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_trade",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

## **BSE Trade Daily Update**

In [5]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_trade_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today's date
today = datetime.today()

date_str = today.strftime("%Y%m%d")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"trade_{file_date}.csv"
)

try:

    url = (
        "https://www.bseindia.com/download/"
        "BhavCopy/Equity/"
        f"BhavCopy_BSE_CM_0_0_0_{date_str}_F_0000.CSV"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    if response.status_code != 200:
        raise Exception("No data yet")

    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # keep BSE cash market stocks only
    df = df[
        (df["Sgmt"] == "CM") &
        (df["FinInstrmTp"] == "STK")
    ]

    # keep required columns
    df = df[[
        "TradDt",
        "TckrSymb",
        "OpnPric",
        "HghPric",
        "LwPric",
        "ClsPric",
        "TtlTradgVol"
    ]]

    # rename columns
    df.columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_trade",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

SUCCESS
Rows: 4868
File saved


In [1]:
print("daily update completed")

daily update completed
